In [5]:
# pip install pyarrow

In [1]:
import timm
import torch
import os 
import pandas as pd
import numpy as np
import time
import json
import threading

from tqdm.notebook import tqdm
from torch import nn
from torch.optim import Adam
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# DataFrame Loader

In [2]:
class CustomDataFrameLoader(torch.utils.data.Dataset): 
    
    def __init__(self, dataframe, classes, transform=None):
        self.dataframe = dataframe
        if type(self.dataframe) is pd.DataFrame :  
            self.data_dict = self.dataframe.to_dict('records')
        else : 
            raise TypeError("ลองรับเฉพาะ dataframe ครับ")
        self.transform = transform
        self.classes = classes

    def __len__(self):
        
        return len(self.data_dict)

    def __getitem__(self, index):

        row = self.data_dict[index]
        img_path = row['image_path']
        image = Image.open(img_path).convert('RGB')
        label = row['label']
        label = torch.tensor(int(self.classes.index(label)))
        if self.transform:
            image = self.transform(image) 
        return {'tensor_image': image, 'tensor_label': label, 'annotation': row}
    
def custom_collate(batch):
    
    image, label = zip(*batch)
    images = [torch.tensor(np.array(img)) for img in image]
    masks = [torch.tensor(np.array(mask)) for mask in label]
    
    return torch.stack(image), torch.stack(label)

In [3]:
df = pd.read_parquet('hymenoptera_data_annotation.parquet')
df.rename(columns={
    'dir': 'image_path', 
    'class': 'label'
}, inplace=True)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_path  398 non-null    object
 1   file_name   398 non-null    object
 2   dataset     398 non-null    object
 3   label       398 non-null    object
 4   class_id    398 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 15.7+ KB


,image_path,file_name,dataset,label,class_id
0,hymenoptera_data/train/bees/2638074627_6b3ae74...,2638074627_6b3ae746a0.jpg,train,bees,0
1,hymenoptera_data/train/bees/507288830_f46e8d4c...,507288830_f46e8d4cb2.jpg,train,bees,0
2,hymenoptera_data/train/bees/2405441001_b06c36f...,2405441001_b06c36fa72.jpg,train,bees,0
3,hymenoptera_data/train/bees/2962405283_22718d9...,2962405283_22718d9617.jpg,train,bees,0
4,hymenoptera_data/train/bees/446296270_d9e8b93e...,446296270_d9e8b93ecf.jpg,train,bees,0


In [4]:
batch_size = 5
classes = ['bees', 'ants']
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.5), 
            (0.5)
        ),        
    ]
)

In [5]:
train_dataset = CustomDataFrameLoader(
    dataframe=df[df['dataset']=='train'], 
    transform=transform, 
    classes=classes
)

In [6]:
train_dataset[1]

{'tensor_image': tensor([[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -0.9922, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]],
 
         [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -0.9922, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]],
 
         [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-

In [7]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True
)

In [8]:
data = next(iter(train_loader))
data

{'tensor_image': tensor([[[[-0.5608, -0.5529, -0.5529,  ..., -0.6941, -0.7020, -0.7333],
           [-0.5922, -0.5608, -0.5451,  ..., -0.7098, -0.7020, -0.7098],
           [-0.5529, -0.5373, -0.5137,  ..., -0.7255, -0.7176, -0.7333],
           ...,
           [-0.1216, -0.1529, -0.2157,  ..., -0.8353, -0.8353, -0.8118],
           [-0.1608, -0.2000, -0.2549,  ..., -0.8353, -0.8275, -0.8118],
           [-0.2078, -0.2471, -0.2706,  ..., -0.8196, -0.8039, -0.8039]],
 
          [[-0.3804, -0.3961, -0.3961,  ..., -0.1451, -0.1686, -0.1843],
           [-0.3804, -0.3804, -0.4118,  ..., -0.1529, -0.1765, -0.2000],
           [-0.3961, -0.3961, -0.4039,  ..., -0.1765, -0.1843, -0.2078],
           ...,
           [-0.2235, -0.2471, -0.2627,  ..., -0.7333, -0.7098, -0.6863],
           [-0.2941, -0.3020, -0.3255,  ..., -0.7176, -0.6941, -0.6941],
           [-0.3255, -0.3490, -0.3725,  ..., -0.7098, -0.6941, -0.6784]],
 
          [[-0.7961, -0.8196, -0.8118,  ..., -0.9843, -0.9843, -0.9922

In [9]:
data['tensor_image'].shape

torch.Size([5, 3, 224, 224])

In [10]:
data['tensor_label']

tensor([0, 1, 1, 1, 0])

In [11]:
data['annotation']

{'image_path': ['hymenoptera_data/train/bees/457457145_5f86eb7e9c.jpg',
  'hymenoptera_data/train/ants/1225872729_6f0856588f.jpg',
  'hymenoptera_data/train/ants/0013035.jpg',
  'hymenoptera_data/train/ants/28847243_e79fe052cd.jpg',
  'hymenoptera_data/train/bees/2053200300_8911ef438a.jpg'],
 'file_name': ['457457145_5f86eb7e9c.jpg',
  '1225872729_6f0856588f.jpg',
  '0013035.jpg',
  '28847243_e79fe052cd.jpg',
  '2053200300_8911ef438a.jpg'],
 'dataset': ['train', 'train', 'train', 'train', 'train'],
 'label': ['bees', 'ants', 'ants', 'ants', 'bees'],
 'class_id': tensor([0, 1, 1, 1, 0])}

# Directory Loader

In [12]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root='./hymenoptera_data/train/', 
    transform=transform
)
train_loader = DataLoader(
    train_dataset, 
    batch_size=5, 
    shuffle=True
)

In [13]:
data = next(iter(train_loader))
data

[tensor([[[[0.3843, 0.3843, 0.3882,  ..., 0.5255, 0.5216, 0.5176],
           [0.3843, 0.3843, 0.3882,  ..., 0.5098, 0.5137, 0.5137],
           [0.3922, 0.3961, 0.4000,  ..., 0.5137, 0.5216, 0.5216],
           ...,
           [0.9176, 0.9020, 0.8353,  ..., 0.5490, 0.5137, 0.4941],
           [0.9176, 0.9176, 0.8745,  ..., 0.5686, 0.5333, 0.5216],
           [0.9098, 0.9255, 0.8980,  ..., 0.5843, 0.5569, 0.5333]],
 
          [[0.5020, 0.5020, 0.4980,  ..., 0.5961, 0.5922, 0.5961],
           [0.5176, 0.5137, 0.5098,  ..., 0.6078, 0.6157, 0.6196],
           [0.5216, 0.5216, 0.5176,  ..., 0.6078, 0.6157, 0.6275],
           ...,
           [0.9451, 0.9137, 0.8353,  ..., 0.6314, 0.6196, 0.5961],
           [0.9529, 0.9451, 0.8863,  ..., 0.6471, 0.6353, 0.6118],
           [0.9490, 0.9529, 0.9216,  ..., 0.6588, 0.6471, 0.6314]],
 
          [[0.2039, 0.2039, 0.2078,  ..., 0.4039, 0.4039, 0.4078],
           [0.2196, 0.2196, 0.2196,  ..., 0.4118, 0.4196, 0.4235],
           [0.2314, 0.23

In [14]:
images_batch = data[0]
images_batch.shape

torch.Size([5, 3, 224, 224])

In [15]:
label_batch = data[1]
label_batch

tensor([1, 0, 0, 0, 0])

In [16]:
label_batch[1].item()

0